In [1]:
import torch
import logging
from omegaconf import OmegaConf

logging.basicConfig(level=logging.INFO)
device = "cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu"
print(f"Device: {device}")

Device: mps


In [2]:
config = OmegaConf.create({
    # Architecture
    "layers": [784, 100, 100, 10],
    "loss_fn": "cross_entropy",

    # CL setting
    "setting": "ClassILMNIST5Task",
    "num_tasks": 5,
    "classes_per_task": 2,

    # Training
    "lr": 1e-4,
    "epochs": 20,
    "batch_size": 256,
    "optimizer": "Adam",
    "scheduler": None,
    "seed": 42,
    "num_workers": 0,
    "device": "cpu",

    # Output
    "output_dir": "outputs/csqn_test",
    "save": False,

    # EWC importance (= λ in the paper)
    "importance_ewc": 1e4,

    # CSQN-specific
    "csqn_M": 10,
    "csqn_method": "sr1",       # "sr1" or "bfgs"
    "csqn_reduce": None,        # None, "ct", or "mrt"
    "csqn_epsilon": 1e-4,
    "csqn_kappa": 1e-12,

    # Dataloader
    "flatten_imgs": "default",
    "use_cnn_encoder": False,

    # Peak model
    "peak": False,
})

In [3]:
from src.dataloaders_2 import ClassILMNIST5Task

dataloader = ClassILMNIST5Task(config)
tasks_dataloaders = dataloader.get_all_tasks_dataloaders()
print(f"Loaded {len(tasks_dataloaders)} tasks")

DataLoader using device: cpu
Loaded 5 tasks


In [4]:
from networks.CSQN_network import CSQN_network

model = CSQN_network(config)
print(model)
print(f"Parameters: {sum(p.numel() for p in model.parameters()):,}")

CSQN_network(
  (layers): ModuleList(
    (0): BP_layer(
      (activation_fn): Softplus(
        (softplus): Softplus(beta=1, threshold=20)
        (sigmoid): Sigmoid()
      )
      (feedforward): Sequential(
        (0): Linear(in_features=784, out_features=100, bias=True)
        (1): Softplus(
          (softplus): Softplus(beta=1, threshold=20)
          (sigmoid): Sigmoid()
        )
      )
    )
    (1): BP_layer(
      (activation_fn): Softplus(
        (softplus): Softplus(beta=1, threshold=20)
        (sigmoid): Sigmoid()
      )
      (feedforward): Sequential(
        (0): Linear(in_features=100, out_features=100, bias=True)
        (1): Softplus(
          (softplus): Softplus(beta=1, threshold=20)
          (sigmoid): Sigmoid()
        )
      )
    )
    (2): BP_layer(
      (activation_fn): Linear()
      (feedforward): Sequential(
        (0): Linear(in_features=100, out_features=10, bias=True)
        (1): Linear()
      )
    )
  )
  (loss_fn): CrossEntropyLoss()
)

In [5]:
from src.trainers import TrainerCL

trainer = TrainerCL(model, tasks_dataloaders, config)
trainer.train()

/opt/homebrew/anaconda3/envs/dev/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
INFO:src.trainers:Training:
 - layers: [784, 100, 100, 10]
 - loss_fn: cross_entropy
 - setting: ClassILMNIST5Task
 - num_tasks: 5
 - classes_per_task: 2
 - lr: 0.0001
 - epochs: 20
 - batch_size: 256
 - optimizer: Adam
 - scheduler: None
 - seed: 42
 - num_workers: 0
 - device: cpu
 - output_dir: outputs/csqn_test
 - save: False
 - importance_ewc: 10000.0
 - csqn_M: 10
 - csqn_method: sr1
 - csqn_reduce: None
 - csqn_epsilon: 0.0001
 - csqn_kappa: 1e-12
 - flatten_imgs: default
 - use_cnn_encoder: False
 - peak: False
 - in_channels: 1
 - model: CSQN_network



Training:
 - layers: [784, 100, 100, 10]
 - loss_fn: cross_entropy
 - setting: ClassILMNIST5Task
 - num_tasks: 5
 - classes_per_task: 2
 - lr: 0.0001
 - epochs: 20
 - batch_size: 256
 - optimizer: Adam
 - scheduler: None
 - seed: 42
 - num_workers: 0
 - device: cpu
 - output_dir: outputs/csqn_test
 - save: False
 - importance_ewc: 10000.0
 - csqn_M: 10
 - csqn_method: sr1
 - csqn_reduce: None
 - csqn_epsilon: 0.0001
 - csqn_kappa: 1e-12
 - flatten_imgs: default
 - use_cnn_encoder: False
 - peak: False
 - in_channels: 1
 - model: CSQN_network



INFO:src.trainers:Starting Task 1/5


Starting Task 1/5


Training epoch 001/20: 100%|██████████| 50/50 [00:00<00:00, 326.75batch/s]
INFO:src.callbacks:				Epoch: 001, Train loss: 0.1803 & test losses: T0: 0.0000 & accuracy: T0: 99.57, Full: 99.57


...

INFO:src.trainers:Testing on all seen classes up to Task 5


Testing on all seen classes up to Task 5


INFO:src.trainers:Seen Classes - Loss: 0.0000, Accuracy: 19.0200


Seen Classes - Loss: 0.0000, Accuracy: 19.0200


SQN (SR1): 100%|██████████| 10/10 [00:00<00:00, 10.30it/s]
